# 🔬 Mini Project 3B — Pekan 1: Setup & Data Collection
**Gas Optimization Framework untuk Smart Contract Solidity**

Jalankan sel dari atas ke bawah secara berurutan.

| Sel | Isi |
|---|---|
| 1 | Verifikasi environment |
| 2 | Install & test solc |
| 3–5 | Setup & test Hardhat |
| 6–8 | Download kontrak dari Etherscan |
| 9–10 | Validasi dataset |
| 11–13 | AST Parser |
| 14 | Checklist akhir |

---
## ✅ Sel 1 — Verifikasi Environment

In [ ]:
import sys, subprocess, os, json
from pathlib import Path
from collections import Counter, defaultdict

ROOT        = Path(os.getcwd()).parent
HARDHAT_DIR = ROOT / 'hardhat_project'
DATASET_DIR = ROOT / 'dataset'          # dir baru (dataset/{domain}/)
RESULTS_DIR = ROOT / 'results'
RESULTS_DIR.mkdir(exist_ok=True)

def run(cmd, cwd=None):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd=cwd)
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.returncode != 0 and result.stderr.strip():
        print('STDERR:', result.stderr.strip())
    return result

print('=== VERIFIKASI ENVIRONMENT ===')
print(f'Python  : {sys.version.split()[0]}')
run('node --version')
run('npm --version')
print(f'ROOT    : {ROOT}')
print('✅ Environment OK')

---
## 🔧 Sel 2 — Install & Test solc 0.8.20

In [2]:
import solcx

# Install jika belum ada
installed = solcx.get_installed_solc_versions()
if solcx.main.Version('0.8.20') not in installed:
    print('⏳ Menginstall solc 0.8.20...')
    solcx.install_solc('0.8.20')

solcx.set_solc_version('0.8.20')
print(f'✅ solc aktif: {solcx.get_solc_version()}')

# ── Test compile kontrak sederhana
SOURCE_TEST = """
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.20;

contract HelloTest {
    uint256 public nilai = 42;
    function getNilai() external view returns (uint256) {
        return nilai;
    }
}
"""

hasil = solcx.compile_source(
    SOURCE_TEST,
    output_values=['abi', 'bin', 'ast'],
    solc_version='0.8.20'
)
print('✅ Test compile berhasil')
print(f'   AST tersedia : {"ast" in list(hasil.values())[0]}')

✅ solc aktif: 0.8.20
✅ Test compile berhasil
   AST tersedia : True


---
## 🔧 Sel 3 — Setup Project Hardhat
> Hanya perlu dijalankan SEKALI. Skip jika `hardhat_project/node_modules/` sudah ada.

In [3]:
node_modules = HARDHAT_DIR / 'node_modules'

if node_modules.exists():
    print('⏭️  node_modules sudah ada — skip install')
else:
    print('⏳ Install Hardhat (1–2 menit)...')
    run('npm init -y', cwd=HARDHAT_DIR)
    run('npm install --save-dev hardhat @nomicfoundation/hardhat-toolbox',
        cwd=HARDHAT_DIR)

# Verifikasi
r = run('npx hardhat --version', cwd=HARDHAT_DIR)
if r.returncode == 0:
    print('✅ Hardhat siap')
else:
    print('❌ Hardhat gagal — cek error di atas')

⏭️  node_modules sudah ada — skip install
2.28.6
✅ Hardhat siap


---
## 🔧 Sel 4 — Tulis Kontrak Test & Script Gas

In [4]:
# ── hardhat.config.js  (CJS — Hardhat 2.x)
hardhat_config = """require("@nomicfoundation/hardhat-toolbox");

module.exports = {
  solidity: {
    version: "0.8.20",
    settings: {
      optimizer: { enabled: false },  // WAJIB false
    },
  },
  networks: { hardhat: {} },
};
"""

# ── TestGas.sol — demonstrasi Redundant SLOAD
test_sol = """// SPDX-License-Identifier: MIT
pragma solidity ^0.8.20;

contract TestGas {
    uint256 public angka;

    // Anti-pattern: baca storage 3x
    function setBoros(uint256 x) public {
        angka = x;
        if (angka > 0) {        // SLOAD ke-2
            angka = angka + 1;  // SLOAD ke-3
        }
    }

    // Optimal: baca storage 1x
    function setHemat(uint256 x) public {
        uint256 _tmp = x;
        if (_tmp > 0) { _tmp = _tmp + 1; }
        angka = _tmp;
    }
}
"""

# ── ukur_gas.js  (CJS — ethers v6 / hardhat-toolbox v6)
gas_script = """const { ethers } = require("hardhat");

async function main() {
  const F = await ethers.getContractFactory("TestGas");
  const c = await F.deploy();
  await c.waitForDeployment();           // ethers v6

  const t1 = await c.setBoros(5); const r1 = await t1.wait();
  const t2 = await c.setHemat(5); const r2 = await t2.wait();

  const diff    = r1.gasUsed - r2.gasUsed;  // BigInt arithmetic
  const pctSave = (Number(diff) * 100 / Number(r1.gasUsed)).toFixed(2);

  console.log("=== HASIL PENGUKURAN GAS ===");
  console.log("setBoros  :", r1.gasUsed.toString());
  console.log("setHemat  :", r2.gasUsed.toString());
  console.log("Selisih   :", diff.toString(), "gas");
  console.log("Penghematan:", pctSave + "%");
}
main().catch(e => { console.error(e); process.exitCode = 1; });
"""

# ── Tulis semua file
(HARDHAT_DIR / 'hardhat.config.js').write_text(hardhat_config)
(HARDHAT_DIR / 'contracts').mkdir(exist_ok=True)
(HARDHAT_DIR / 'scripts').mkdir(exist_ok=True)
(HARDHAT_DIR / 'contracts' / 'TestGas.sol').write_text(test_sol)
(HARDHAT_DIR / 'scripts'   / 'ukur_gas.js').write_text(gas_script)

print('✅ hardhat.config.js  ditulis (CJS)')
print('✅ contracts/TestGas.sol ditulis')
print('✅ scripts/ukur_gas.js   ditulis (CJS, ethers v6)')

✅ hardhat.config.js  ditulis (CJS)
✅ contracts/TestGas.sol ditulis
✅ scripts/ukur_gas.js   ditulis (CJS, ethers v6)


---
## 🔧 Sel 5 — Compile & Ukur Gas (Test Hardhat)

In [5]:
print('⏳ Compile...')
run('npx hardhat compile', cwd=HARDHAT_DIR)

print('\n⏳ Ukur gas...')
run('npx hardhat run scripts/ukur_gas.js --network hardhat', cwd=HARDHAT_DIR)

# Output yang diharapkan:
# === HASIL PENGUKURAN GAS ===
# setBoros  : 4XXXX
# setHemat  : 4XXXX
# Selisih   : XXXX gas
# Penghematan: X.XX%

⏳ Compile...
Nothing to compile

⏳ Ukur gas...
=== HASIL PENGUKURAN GAS ===
setBoros  : 44252
setHemat  : 24025
Selisih   : 20227 gas
Penghematan: 45.71%


CompletedProcess(args='npx hardhat run scripts/ukur_gas.js --network hardhat', returncode=0, stdout='=== HASIL PENGUKURAN GAS ===\nsetBoros  : 44252\nsetHemat  : 24025\nSelisih   : 20227 gas\nPenghematan: 45.71%\n', stderr='')

---
## 📥 Sel 6 — Load API Key Etherscan
> Pastikan file `.env` di root project sudah diisi:
> ```
> ETHERSCAN_API_KEY=isi_api_key_kamu
> ```

In [6]:
import requests, json, time
from collections import Counter, defaultdict
from dotenv import load_dotenv

# Baca .env dari root project
load_dotenv(ROOT / '.env')
API_KEY = os.getenv('ETHERSCAN_API_KEY', '')

if not API_KEY:
    raise ValueError('❌ ETHERSCAN_API_KEY tidak ditemukan. Isi file .env di root project.')

print(f'✅ API Key loaded: {API_KEY[:6]}{"*" * 10}')
print(f'📁 Dataset dir  : {DATASET_DIR}')

✅ API Key loaded: SNQRM9**********
📁 Dataset dir  : C:\Users\Ridho\project\KBJ\gas_optimization\contracts_dataset


---
## 📥 Sel 7 — Daftar 50 Alamat Kontrak

In [ ]:
import requests, time

# ── Load konfigurasi dataset dari contracts_selection.json
# File ini dibuat oleh dataset_expansion.ipynb dan dapat diganti
# tanpa mengubah notebook (data-driven configuration)
SELECTION_FILE = ROOT / 'contracts_selection.json'

if not SELECTION_FILE.exists():
    raise FileNotFoundError(
        f'contracts_selection.json tidak ditemukan di {ROOT}\n'
        'Jalankan notebooks/dataset_expansion.ipynb terlebih dahulu.'
    )

with open(SELECTION_FILE) as f:
    selection = json.load(f)

per_domain_sel = defaultdict(list)
for c in selection:
    per_domain_sel[c['domain']].append(c)

total = len(selection)
print(f'📋 Loaded: {SELECTION_FILE.name}')
print(f'   Total kontrak: {total}')
for domain in sorted(per_domain_sel):
    contracts = per_domain_sel[domain]
    cx_str = ', '.join(
        f'{lv}={sum(1 for c in contracts if c["complexity"]==lv)}'
        for lv in ['Simple','Medium','Complex']
        if any(c['complexity']==lv for c in contracts)
    )
    print(f'   {domain:<12}: {len(contracts)} kontrak ({cx_str})')

print(f'\n💡 Untuk mengganti dataset: edit contracts_selection.json')
print(f'   (tanpa perlu mengubah notebook ini)')

---
## 📥 Sel 8 — Download Kontrak dari Etherscan

In [ ]:
LOG_BERHASIL = []
LOG_GAGAL    = []
LOG_SKIP     = []

def download_kontrak(entry):
    """Download satu kontrak dari Etherscan jika belum ada. Idempotent."""
    alamat = entry['alamat']
    nama   = entry['nama']
    dest   = Path(entry['file'])

    if dest.exists():
        LOG_SKIP.append(nama)
        return True

    dest.parent.mkdir(parents=True, exist_ok=True)

    url = (f'https://api.etherscan.io/v2/api?chainid=1'
           f'&module=contract&action=getsourcecode'
           f'&address={alamat}&apikey={API_KEY}')
    try:
        resp = requests.get(url, timeout=15)
        data = resp.json()
    except Exception as e:
        print(f'  ❌ {nama}: {e}')
        LOG_GAGAL.append({'nama': nama, 'alasan': str(e)})
        return False

    if data.get('status') != '1':
        msg = data.get('message', 'API error')
        print(f'  ❌ {nama} [{alamat[:10]}...]: {msg}')
        LOG_GAGAL.append({'nama': nama, 'alasan': msg})
        return False

    src = data['result'][0].get('SourceCode', '')
    if not src:
        print(f'  ⚠️  {nama}: tidak verified')
        LOG_GAGAL.append({'nama': nama, 'alasan': 'not verified'})
        return False

    with open(dest, 'w', encoding='utf-8') as f:
        if src.startswith('{{'):
            try:
                fj = json.loads(src[1:-1])
                parts = []
                for path, content in fj['sources'].items():
                    parts.append(f'// === File: {path} ===')
                    parts.append(content['content'])
                f.write('\n\n'.join(parts))
            except Exception:
                f.write(src)
        else:
            f.write(src)

    print(f'  ✅ Downloaded: {nama}')
    LOG_BERHASIL.append(nama)
    return True


print('🔍 Memeriksa file kontrak...\n')
for entry in selection:
    download_kontrak(entry)
    if entry['nama'] not in LOG_SKIP:
        time.sleep(0.25)

print(f'\n{"="*55}')
print(f'✅ Sudah ada (skip) : {len(LOG_SKIP):3d} kontrak')
print(f'✅ Baru didownload  : {len(LOG_BERHASIL):3d} kontrak')
print(f'❌ Gagal            : {len(LOG_GAGAL):3d} kontrak')
if LOG_GAGAL:
    print('\nKontrak gagal:')
    for g in LOG_GAGAL:
        print(f'  {g["nama"]:35s} {g.get("alasan","")}')

In [ ]:
# Sel 8b tidak lagi diperlukan — Sel 8 sekarang otomatis skip kontrak yang sudah ada
# dan download hanya yang belum tersedia. Dataset dikontrol via contracts_selection.json.
print('ℹ️  Sel 8b: tidak diperlukan — Sel 8 sudah idempotent.')

---
## 📊 Sel 9 — Validasi Dataset

In [ ]:
# Validasi dataset dari contracts_selection.json
print('📊 RINGKASAN DATASET')
print('=' * 60)

for domain in sorted(per_domain_sel.keys()):
    contracts = per_domain_sel[domain]
    locs = [c['loc'] for c in contracts]
    cx   = Counter(c['complexity'] for c in contracts)
    print(f'\n{domain}: {len(contracts)} kontrak')
    print(f'  LOC      : min={min(locs)}, max={max(locs)}, avg={sum(locs)//len(locs)}')
    print(f'  Complexity: {dict(cx)}')

print('\n📐 Distribusi Complexity (total):')
all_cx = Counter(c['complexity'] for c in selection)
for level in ['Simple', 'Medium', 'Complex']:
    count = all_cx.get(level, 0)
    bar   = '█' * min(count, 40)
    print(f'  {level:<8}: {bar} ({count})')
print(f'\n  Total: {len(selection)} kontrak | {len(per_domain_sel)} domain')
print(f'  Note: Token & NFT tidak memiliki Simple contract (ERC standard inherently complex)')

---
## 🔍 Sel 10 — Verifikasi Compile

In [ ]:
import re, tempfile, solcx
from packaging.version import Version as PkgVersion

SOLC_MAP = {'0.4':'0.4.26','0.5':'0.5.17','0.6':'0.6.12','0.7':'0.7.6','0.8':'0.8.20'}

def detect_solc_ver(source):
    m = re.search(r'pragma solidity\s*=?\s*(\d+\.\d+\.\d+)\s*;', source)
    if m: return m.group(1)
    m = re.search(r'pragma solidity\s+[>=^<~]*\s*(\d+\.\d+)', source)
    if not m: return '0.8.20'
    return SOLC_MAP.get('.'.join(m.group(1).split('.')[:2]), '0.8.20')

def dedup_pragma(source):
    seen, lines = False, []
    for line in source.splitlines():
        if re.match(r'\s*pragma solidity', line):
            if not seen: lines.append(line); seen = True
        else:
            lines.append(line)
    return '\n'.join(lines)

def parse_multifile(source):
    parts = re.split(r'// === File: (.+?) ===', source)
    if len(parts) <= 1: return None
    files = {}
    for i in range(1, len(parts)-1, 2):
        files[parts[i].strip()] = parts[i+1]
    return files or None

def smart_compile_ast(filepath):
    src = Path(filepath).read_text(encoding='utf-8')
    if 'pragma solidity' not in src: return None
    ver = detect_solc_ver(src)
    try:
        if PkgVersion(ver) < PkgVersion('0.4.11'): ver = '0.4.26'
    except Exception: pass
    installed = [str(v) for v in solcx.get_installed_solc_versions()]
    if ver not in installed:
        print(f'    ⏳ install solc {ver}...')
        solcx.install_solc(ver)
    files = parse_multifile(src)
    if files:
        try:
            inp = {"language":"Solidity","sources":{n:{"content":c} for n,c in files.items()},
                   "settings":{"optimizer":{"enabled":False},"outputSelection":{"*":{"":["ast"]}}}}
            out = solcx.compile_standard(inp, solc_version=ver)
            return out['sources'][next(iter(out['sources']))]['ast']
        except Exception: pass
        try:
            with tempfile.TemporaryDirectory() as td:
                tdp = Path(td)
                for name, content in files.items():
                    fp = tdp/name; fp.parent.mkdir(parents=True,exist_ok=True)
                    fp.write_text(content,encoding='utf-8')
                out = solcx.compile_files([str(tdp/next(iter(files)))],output_values=['ast'],
                    solc_version=ver,optimize=False,allow_paths=[td])
                if out: return list(out.values())[0]['ast']
        except Exception: pass
    else:
        src_clean = dedup_pragma(src)
        try:
            out = solcx.compile_source(src_clean,output_values=['ast'],solc_version=ver,optimize=False)
            return list(out.values())[0]['ast']
        except Exception: pass
    try:
        main_src = src if not files else list(files.values())[0]
        stripped = re.sub(r'^\s*import\s+[^;]+;','',main_src,flags=re.MULTILINE)
        stripped = dedup_pragma(stripped)
        out = solcx.compile_source(stripped,output_values=['ast'],solc_version=ver,optimize=False)
        if out: return list(out.values())[0]['ast']
    except Exception: pass
    return None

# Verifikasi compile pada selection
print('🔍 Verifikasi compile...\n')
bisa_compile  = []
gagal_compile = []

for m in selection:
    ast = smart_compile_ast(m['file'])
    if ast:
        bisa_compile.append(m)
    else:
        ver = detect_solc_ver(Path(m['file']).read_text(encoding='utf-8'))
        print(f'  ⚠️  {m["nama"]:<35} [solc {ver}]')
        gagal_compile.append(m)

print(f'\n{"="*55}')
print(f'✅ Bisa compile : {len(bisa_compile)}/{len(selection)}')
print(f'⚠️  Gagal compile: {len(gagal_compile)}/{len(selection)}')

---
## 🌳 Sel 11 — Definisi Fungsi AST Parser

In [12]:
def generate_ast(filepath):
    """Generate AST dari file Solidity. Return dict atau None."""
    return smart_compile_ast(filepath)


def walk_ast(node, callback, depth=0):
    """Rekursif jelajahi semua node AST."""
    if not isinstance(node, dict):
        return
    callback(node, depth)
    for value in node.values():
        if isinstance(value, dict):
            walk_ast(value, callback, depth + 1)
        elif isinstance(value, list):
            for item in value:
                if isinstance(item, dict):
                    walk_ast(item, callback, depth + 1)


def find_nodes(ast, target_type):
    """Kembalikan list semua node dengan nodeType tertentu."""
    result = []
    walk_ast(ast, lambda n, d: result.append(n)
             if n.get('nodeType') == target_type else None)
    return result


print('✅ Fungsi AST siap: generate_ast(), walk_ast(), find_nodes()')

✅ Fungsi AST siap: generate_ast(), walk_ast(), find_nodes()


---
## 🌳 Sel 12 — Test AST pada 1 Kontrak

In [13]:
target = next((m for m in metadata if m.get('compile_ok')), None)

if not target:
    print('❌ Tidak ada kontrak yang bisa compile')
else:
    ast = generate_ast(target['file'])
    if ast:
        functions = find_nodes(ast, 'FunctionDefinition')
        print(f'✅ AST berhasil: {target["nama"]}')
        print(f'   Root node  : {ast.get("nodeType")}')
        print(f'   Child nodes: {len(ast.get("nodes", []))}')
        print(f'   Functions  : {len(functions)}')
        for fn in functions[:5]:
            name = fn.get('name') or '(fallback/receive)'
            vis  = fn.get('visibility', '-')
            print(f'     • {name:<30} visibility={vis}')
    else:
        print('❌ generate_ast() gagal')

✅ AST berhasil: WETH9
   Root node  : None
   Child nodes: 0
   Functions  : 0


---
## 🌳 Sel 13 — Inventarisasi Node Types

In [14]:
kontrak_valid = [m for m in metadata if m.get('compile_ok')]
print(f'Inventarisasi pada {len(kontrak_valid)} kontrak...\n')

total_types = Counter()
for m in kontrak_valid:
    ast = generate_ast(m['file'])
    if ast:
        counter = Counter()
        walk_ast(ast, lambda n, d: counter.update(
            [n['nodeType']] if 'nodeType' in n else []))
        total_types.update(counter)

print(f'{"Node Type":<40} {"Count":>8}')
print('─' * 50)
for tipe, jumlah in total_types.most_common(20):
    print(f'{tipe:<40} {jumlah:>8,}')

# Node relevan untuk 6 anti-pattern
print('\n🎯 Node relevan untuk Pekan 2:')
TARGET_NODES = [
    ('Identifier',         'Redundant SLOAD'),
    ('ForStatement',       'Unoptimized Loop'),
    ('ElementaryTypeName', 'String vs Bytes32'),
    ('FunctionDefinition', 'Public vs External + Dead Code'),
    ('BinaryOperation',    'Unchecked Arithmetic'),
    ('UncheckedStatement', 'Unchecked Arithmetic (scope aman)'),
]
for node, keterangan in TARGET_NODES:
    count  = total_types.get(node, 0)
    status = '✅' if count > 0 else '⚠️ '
    print(f'  {status} {node:<25} ({count:,}x) — {keterangan}')

Inventarisasi pada 46 kontrak...

Node Type                                   Count
──────────────────────────────────────────────────
Identifier                                  1,615
ElementaryTypeName                            851
VariableDeclaration                           835
FunctionCall                                  478
ParameterList                                 454
MemberAccess                                  341
ExpressionStatement                           337
Literal                                       291
Block                                         250
BinaryOperation                               237
FunctionDefinition                            195
StructuredDocumentation                       144
Assignment                                    130
IdentifierPath                                111
VariableDeclarationStatement                  109
IndexAccess                                    97
UserDefinedTypeName                            75
Return         

---
## ✅ Sel 14 — Checklist Akhir Pekan 1

In [ ]:
hh_ok = run('npx hardhat --version', cwd=HARDHAT_DIR).returncode == 0

n_files_exist = sum(1 for c in selection if Path(c['file']).exists())
n_domains     = len(per_domain_sel)

checks = [
    ('solc 0.8.20 aktif',
        solcx.get_solc_version() is not None),
    ('Hardhat berjalan',
        hh_ok),
    ('contracts_selection.json ada',
        (ROOT / 'contracts_selection.json').exists()),
    (f'Minimal 70/{len(selection)} kontrak filenya ada',
        n_files_exist >= 70),
    ('Semua 5 domain ada',
        n_domains == 5),
    ('Minimal 60 kontrak bisa compile',
        len(bisa_compile) >= 60),
    ('AST parser berjalan',
        bool(kontrak_valid) and generate_ast(kontrak_valid[0]['file']) is not None),
    ('Node types sudah diinventarisasi',
        len(total_types) > 0),
]

print('=' * 50)
print('  CHECKLIST AKHIR PEKAN 1')
print('=' * 50)

semua_ok = True
for label, ok in checks:
    icon = '✅' if ok else '❌'
    if not ok: semua_ok = False
    print(f'  {icon} {label}')

print('─' * 50)
print(f'  Dataset     : {len(selection)} kontrak ({n_domains} domain)')
print(f'  Files exist : {n_files_exist}/{len(selection)}')
print(f'  Compile OK  : {len(bisa_compile)}/{len(selection)}')
print('=' * 50)
print('  🎉 Siap masuk Pekan 2!' if semua_ok
      else '  ⚠️  Cek item ❌ di atas')
print('=' * 50)